In [1]:
import torch
import evaluate
import numpy as np
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import DataCollatorForTokenClassification
from transformers import TrainingArguments, Trainer
from huggingface_hub import notebook_login

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [3]:
interleaved_dataset = load_from_disk("data/processed/harmonized_datasets/interleaved_dataset")

interleaved_dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 72000
    })
    validation: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 3699
    })
    test: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 7374
    })
})

In [4]:
tokenized_dataset = load_from_disk("data/processed/tokenized_datasets/tokenized_dataset")

tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 72000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3699
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 7374
    })
})

In [5]:
model_id = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [7]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [8]:
[print(len(tokenized_dataset['train']['labels'][i])) for i in range(3)];

67
84
35


In [9]:
batch = data_collator([tokenized_dataset["train"][i] for i in range(3)])
batch["labels"]

tensor([[-100,    0, -100, -100, -100,    0, -100,    0,    3, -100, -100,    4,
            4, -100, -100,    4,    0,    3, -100,    0,    0, -100,    0, -100,
            0,    3, -100,    4,    0, -100,    0,    0, -100, -100,    0,    0,
         -100, -100,    0,    0,    0, -100, -100, -100,    0,    0,    1,    2,
            0, -100,    0,    3, -100,    4,    0, -100, -100, -100, -100,    0,
            3, -100,    0, -100,    0, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100],
        [-100,    0,    0,    0,    0,    0, -100,    0,    0,    0,    0,    0,
            0,    0,    0, -100,    0,    0,    0,    0,    0,    0,    0,    0,
            0, -100, -100,    0, -100,    5,    0, -100,    0,    0,    0,    0,
            0,    0,    0, -100,    0, -100,    0, -100,    0,    0, -100,    0,
         -100,    0,    0,    0,    0,    0,    0,    0,    0, -100,    0,    0,
            0,    0, -100, 

In [10]:
[print(len(batch['labels'][i])) for i in range(3)];

84
84
84


In [11]:
metric = evaluate.load("seqeval")

In [13]:
label_names = interleaved_dataset['train']['ner'].features.feature.names
label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [14]:
labels = interleaved_dataset["train"][0]["ner"]
labels = [label_names[i] for i in labels]
labels

['O',
 'O',
 'O',
 'B-ORG',
 'I-ORG',
 'I-ORG',
 'I-ORG',
 'O',
 'B-ORG',
 'O',
 'O',
 'O',
 'O',
 'B-ORG',
 'I-ORG',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-PER',
 'I-PER',
 'O',
 'O',
 'B-ORG',
 'I-ORG',
 'O',
 'O',
 'B-ORG',
 'O',
 'O']

In [15]:
predictions = labels.copy()
predictions[2] = "O"
metric.compute(predictions=[predictions], references=[labels])

{'ORG': {'precision': np.float64(1.0),
  'recall': np.float64(1.0),
  'f1': np.float64(1.0),
  'number': np.int64(5)},
 'PER': {'precision': np.float64(1.0),
  'recall': np.float64(1.0),
  'f1': np.float64(1.0),
  'number': np.int64(1)},
 'overall_precision': np.float64(1.0),
 'overall_recall': np.float64(1.0),
 'overall_f1': np.float64(1.0),
 'overall_accuracy': 1.0}

In [ ]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    results = {
        "overall_precision": all_metrics["overall_precision"],
        "overall_recall": all_metrics["overall_recall"],
        "overall_f1": all_metrics["overall_f1"],
        "overall_accuracy": all_metrics["overall_accuracy"],
    }

    for k, v in all_metrics.items():
        if k not in ["overall_precision", "overall_recall", "overall_f1", "overall_accuracy"]:
            results[f"{k}_f1"] = v["f1"]
            results[f"{k}_precision"] = v["precision"]
            results[f"{k}_recall"] = v["recall"]
    
    return results           
    

In [17]:
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}

In [18]:
model = AutoModelForTokenClassification.from_pretrained(
    model_id,
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
model.config.num_labels

9

In [36]:
notebook_login()

In [ ]:
args = TrainingArguments(
    output_dir="xlm-roberta-ner-hun_eng_ger",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=1,
    weight_decay=0.01,
    push_to_hub=False,
    report_to="wandb",
    run_name="ner_exp_01"
)

In [ ]:
eval_datasets = {
    "hun": tokenized_hun_dataset["validation"],
    "eng": tokenized_eng_dataset["validation"],
    "ger": tokenized_ger_dataset["validation"]
}

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=eval_datasets,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

In [43]:
trainer.train()

d:\Projects\hf_practice\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
trainer.push_to_hub(commit_message="Training complete")